# Compare Puppi MET and Raw Puppi MET

In [7]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib
import vector
import math
import numpy as np
import matplotlib.gridspec as gridspec
vector.register_awkward()
from coffea.jetmet_tools import CorrectedJetsFactory, CorrectedMETFactory, JECStack
import coffea
import cachetools
coffea.__version__

'0.7.29'

In [8]:
def met_correction_after_jec(events, METcoll, jets_pre_jec, jets_post_jec):
    '''This function rescale the MET vector by minus delta of the jets after JEC correction
    and before the jEC correction.
    This can be used also to rescale the MET when updating on the fly the JEC calibration. '''
    orig_tot_px = ak.sum(jets_pre_jec.px, axis=1)
    orig_tot_py = ak.sum(jets_pre_jec.py, axis=1)
    new_tot_px = ak.sum(jets_post_jec.px, axis=1)
    new_tot_py = ak.sum(jets_post_jec.py, axis=1)
    newpx =  events[METcoll].px - (new_tot_px - orig_tot_px) 
    newpy =  events[METcoll].py - (new_tot_py - orig_tot_py) 
    
    newMetPhi = np.arctan2(newpy, newpx)
    newMetPt = (newpx**2 + newpy**2)**0.5
    return  {"pt": newMetPt, "phi": newMetPhi}



In [9]:
filename="root://t3dcachedb03.psi.ch:1094//pnfs/psi.ch/cms/trivcat/store/user/mmalucch/DYJetsToLL_M-50_MET_studies/store/mc/RunIII2024Summer24NanoAODv15/DYto2Mu-4Jets_Bin-MLL-50_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v3/2520000/e1d80906-28f6-4da0-9698-dc5cb22c713b.root"

events = NanoEventsFactory.from_root(filename, schemaclass=NanoAODSchema, entry_stop=10000 ).events()

In [10]:
print(events.RawPuppiMET.pt)
print(events.PuppiMET.pt)

[39.5, 41, 17.1, 24.1, 35.1, 24.8, 17.5, ... 17.1, 17, 51.4, 24, 18.1, 2.37, 17.2]
[46.7, 49.3, 24.4, 35, 30.3, 52.9, 6.71, ... 18.9, 62.7, 26.6, 37.2, 2.37, 28.9]


In [12]:
jet_raw = ak.copy(events.Jet)
jet_raw_pt = jet_raw.pt * (
    1 - jet_raw.rawFactor
)  * (1-jet_raw.muonSubtrFactor)
jet_raw_mass = jet_raw.mass * (
    1 - jet_raw.rawFactor
)  * (1-jet_raw.muonSubtrFactor)

jet_raw_noMu = ak.zip(
    {
        "pt": jet_raw_pt,
        "px": jet_raw_pt * np.cos(jet_raw.phi),
        "py": jet_raw_pt * np.sin(jet_raw.phi),
        "phi": jet_raw.phi,
        "eta": jet_raw.eta,
        "mass": jet_raw_mass,
        "pt_orig":jet_raw_pt,
        "pt_raw":jet_raw_pt,
    },
    with_name="Momentum4D",
)
jet_noMu = ak.zip(
    {
        "pt": events.Jet.pt* (1-jet_raw.muonSubtrFactor),
        "px": events.Jet.pt * np.cos(events.Jet.phi),
        "py": events.Jet.pt * np.sin(events.Jet.phi),
        "phi": events.Jet.phi,
        "eta": events.Jet.eta,
        "mass": events.Jet.mass* (1-jet_raw.muonSubtrFactor),
        "pt_orig":jet_raw_pt,
        "pt_raw":jet_raw_pt,
    },
    with_name="Momentum4D",
)




In [ ]:
low_pt_raw_noMU= events.CorrT1METJet